# Introducción

En este cuaderno estaremos extendiendo la funcionalidad de extracción de las coordenadas GPS y el nombre de un fichero que contiene numerosos talleres con el objetivo de geolocalizarlos en un mapa.

Para ello primero importaremos e instalaremos las librerías y dependencias necesarias

In [2]:
import csv
import pandas as pd
import numpy as np
from geopy.geocoders import ArcGIS
import folium
from scipy.spatial import cKDTree
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt

Primeramente cargamos los datos del fichero csv que hemos arrastrado a la carpeta de trabajo

In [3]:
datos = []

csvfile = open("garage_locations.csv", encoding="utf8")
reader = csv.reader(csvfile)

for row in reader:
    datos.append(row)

datos

[['"var optionsMarqueur1 = {position: new google.maps.LatLng(43.6163939,1.4402323),icon: image,map: map};var marqueur1 = new google.maps.Marker(optionsMarqueur1);markers.push(marqueur1);markers_garage.push(marqueur1);var infoBulle1 = new google.maps.InfoWindow({content: contenuInfobulle1});google.maps.event.addListener(marqueur1, ""mouseover"", function() {if (typeof( window.infoopened ) != ""undefined"") { infoopened.close(); }infoBulle1.open(map, marqueur1);infoopened = infoBulle1;});google.maps.event.addListener(marqueur1, ""click"", function() {if (typeof( window.infoopened ) != ""undefined"") { infoopened.close(); }infoBulle1.open(map, marqueur1);infoopened = infoBulle1;});var image = ""images/carre_rouge.png"";var contenuInfobulle2 =  \'<div class=""map_popup"">\' + \'<div class=""left map_popup_txt"">\' + \'<div class=""wrapper_contenu_infobulle"">\' + \'<img class=""left"" src=""images/carre_rouge.png"" alt=""Marqueur Carrosserie""/>\' + \'<h1 class=""left"">\' + \'Carrosserie 

Ahora procederemos a extraer la información del nombre del taller y sus coordenadas GPS de cada uno de los registros del fichero

In [4]:
listacompleta = []

for x in range(0, len(datos)):

  cadena = str(datos[x])

  subcadena1 = r"LatLng("
  subcadena2 = r"),icon:"
  subcadena3 = r"<h1 class="
  subcadena4 = r"\' + \'</h1>\' + \'"

  ilatlong = cadena.find(subcadena1)+7
  flatlong = cadena.find(subcadena2)
  cadena_1 = cadena[ilatlong:flatlong]
  latlong = cadena_1.split(sep=",")

  inombre = cadena.find(subcadena3)+26
  fnombre = cadena.find(subcadena4)
  nombre = cadena[inombre:fnombre]

  fila = [nombre, (latlong[0]), (latlong[1])]

  listacompleta.append(fila)

listacompleta

[['Carrosserie M. SERVICES AUTOMOBILES', '43.6163939', '1.4402323'],
 ['Garage et Carrosserie DELAMAIRE', '43.6778143', '1.4156883'],
 ['Garage MECA SPORT AUTO', '48.5428498', '-2.2062701'],
 ['Garage et Carrosserie AUTO AGM', '43.6203689', '6.972497'],
 ['Carrosserie M. SERVICES AUTOMOBILES', '43.6054951', '6.9926871'],
 ['Garage SILVESTRE ET FILS', '43.6778143', '1.4156883'],
 ['Garage et Carrosserie COMBE SARL MISTRAL ENTREPRISES',
  '44.884817',
  '6.628008899999941'],
 ['Garage MECA SPORT AUTO', '43.8115912', '4.8340044'],
 ['Carrosserie THIMONNIER Benoit', '43.576551', '7.01902'],
 ['Carrosserie SARL MAURIN', '45.981425', '-1.041028'],
 ['Garage et Carrosserie OLIVIER', '45.7059243', '-0.3387318'],
 ['Carrosserie PELAGE BRUNO', '45.370801', '5.992381'],
 ['Garage SAINT-JUSTIN', '44.8631352', '-0.5604802'],
 ['Garage GARCIA HERMANOS', '43.981557', '-0.229919'],
 ['Garage CARROSSERIE DU MIDI', '43.933607', '-0.920563'],
 ['Garage CAZOULS AUTO DISTRIBUTION', '43.5572492', '3.7968754

Ahora procederemos a guardar el listado de talleres en un DataFrame con tres features:

* Nombre del Taller
* Latitud
* Longitud

In [5]:
df = pd.DataFrame()
df = pd.DataFrame(listacompleta, columns=['Nombre del Taller', 'Latitud', 'Longitud'])
df['Nombre del Taller'] = df['Nombre del Taller'].replace({'\\\\':''}, regex=True)
df

,Nombre del Taller,Latitud,Longitud
0,Carrosserie M. SERVICES AUTOMOBILES,43.6163939,1.4402323
1,Garage et Carrosserie DELAMAIRE,43.6778143,1.4156883
2,Garage MECA SPORT AUTO,48.5428498,-2.2062701
3,Garage et Carrosserie AUTO AGM,43.6203689,6.972497
4,Carrosserie M. SERVICES AUTOMOBILES,43.6054951,6.9926871
...,...,...,...
678,Garage Carrosserie GRANGIER,47.9291299,-1.941751599999975
679,Garage BLANCHET Nicolas,47.425029,2.053595
680,Garage AUTO OCCASION SERVICE,47.82355330000001,1.2957106999999723
681,Carrosserie PRECICAR,48.5076567,-4.0748286


Procederemos a limpiar el DataFrame de valores perdidos y reseteamos los índices ya que después vamos a iterar sobre el dataframe.

In [6]:
df1 = pd.DataFrame()
df1 = df.dropna()
df1.reset_index(drop=True, inplace=True)

print(df1.info())

<class 'pandas.DataFrame'>
RangeIndex: 683 entries, 0 to 682
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Nombre del Taller  683 non-null    str  
 1   Latitud            683 non-null    str  
 2   Longitud           683 non-null    str  
dtypes: str(3)
memory usage: 16.1 KB
None


Ahora utilizaremos un geolocalizador para enrriquecer el DataFrame con nuevas features calculadas a partir de las coordenadas GPS.

In [7]:
localización = []

geolocator = ArcGIS(user_agent="Mi_Geolocalizador")

for index, row in df1.iterrows():
  location = geolocator.reverse((row['Latitud'], row['Longitud']), timeout=None)
  localización.append(location.raw)

Antes de añadir las nuevas features al DataFrame comprobamos que la extracción de información haya sido correcta.

In [8]:
print(localización[0])

{'Match_addr': 'Autosur', 'LongLabel': 'Autosur, 51 Boulevard des Minimes, 31200, Toulouse, Haute-Garonne, Occitanie, FRA', 'ShortLabel': 'Autosur', 'Addr_type': 'POI', 'Type': 'Government Office', 'PlaceName': 'Autosur', 'AddNum': '51', 'Address': '51 Boulevard des Minimes', 'Block': '', 'Sector': '', 'Neighborhood': '', 'District': '', 'City': 'Toulouse', 'MetroArea': '', 'Subregion': 'Haute-Garonne', 'Region': 'Occitanie', 'RegionAbbr': 'OCC', 'Territory': '', 'Postal': '31200', 'PostalExt': '', 'CntryName': 'France', 'CountryCode': 'FRA', 'X': 1.44018, 'Y': 43.61625, 'InputX': 1.4402323, 'InputY': 43.6163939, 'StrucType': '', 'StrucDet': ''}


Añadiremos las features seleccionadas al DataFrame. Note que lo estamos haciendo sobre un nuevo DataFrame para dejar una copia intacta del DataFrame original.

In [9]:
Calle = []
Codigo_Postal = []
Ciudad = []
Departamento = []

for i in range(0, len(df1)):
  road=[localización[i]['Address']]
  postcode=[localización[i]['Postal']]
  city=[localización[i]['City']]
  department=[localización[i]['Subregion']]
  Calle.append(road)
  Codigo_Postal.append(postcode)
  Ciudad.append(city)
  Departamento.append(department)

df1['Calle'] = pd.DataFrame(Calle)
df1['Código Postal'] = pd.DataFrame(Codigo_Postal)
df1['Ciudad'] = pd.DataFrame(Ciudad)
df1['Departamento'] = pd.DataFrame(Departamento)
df1['Dirección'] = df1['Calle']+", "+df1['Código Postal']+" "+df1['Ciudad']

df1

,Nombre del Taller,Latitud,Longitud,Calle,Código Postal,Ciudad,Departamento,Dirección
0,Carrosserie M. SERVICES AUTOMOBILES,43.6163939,1.4402323,51 Boulevard des Minimes,31200,Toulouse,Haute-Garonne,"51 Boulevard des Minimes, 31200 Toulouse"
1,Garage et Carrosserie DELAMAIRE,43.6778143,1.4156883,3 Impasse Mozart,31140,Aucamville,Haute-Garonne,"3 Impasse Mozart, 31140 Aucamville"
2,Garage MECA SPORT AUTO,48.5428498,-2.2062701,5 Rue de la Croix des Tilleuls,22130,Créhen,Côtes-d'Armor,"5 Rue de la Croix des Tilleuls, 22130 Créhen"
3,Garage et Carrosserie AUTO AGM,43.6203689,6.972497,,06370,Mouans-Sartoux,Alpes-Maritimes,", 06370 Mouans-Sartoux"
4,Carrosserie M. SERVICES AUTOMOBILES,43.6054951,6.9926871,838 Avenue de Font Roubert,06250,Mougins,Alpes-Maritimes,"838 Avenue de Font Roubert, 06250 Mougins"
...,...,...,...,...,...,...,...,...
678,Garage Carrosserie GRANGIER,47.9291299,-1.941751599999975,Rue de Rennes,35330,La Chapelle-Bouëxic,Ille-et-Vilaine,"Rue de Rennes, 35330 La Chapelle-Bouëxic"
679,Garage BLANCHET Nicolas,47.425029,2.053595,33 Boulevard de la République,41300,Salbris,Loir-et-Cher,"33 Boulevard de la République, 41300 Salbris"
680,Garage AUTO OCCASION SERVICE,47.82355330000001,1.2957106999999723,7 Rue de Chateaudun,41290,Oucques La Nouvelle,Loir-et-Cher,"7 Rue de Chateaudun, 41290 Oucques La Nouvelle"
681,Carrosserie PRECICAR,48.5076567,-4.0748286,7 Boulevard Thierry d'Argenlieu,29400,Landivisiau,Finistère,"7 Boulevard Thierry d'Argenlieu, 29400 Landivi..."


Por último, procedemos a geolocalizar cada uno de los talleres en un mapa, añadiendo un popup con la dirección del mismo.

In [10]:
map1 = folium.Map(location=[47.6163939, 2.4402323], zoom_start=6)

tooltip = "Haz clic para ver el nombre del taller y la dirección"

df1.apply(lambda row:folium.Marker(location=[row["Latitud"], row["Longitud"]], popup=(row["Nombre del Taller"] + " " + row['Dirección']), tooltip=tooltip).add_to(map1), axis=1)
map1

In [25]:
# Ingresamos la dirección donde se nos ha averiado el coche. Para este ejemplo ingresaremos Aeropuerto de Pau-Pirineos aunque puedes elegir la que quieras siempre y cuando tenga talleres "cercanos"

direccion = input('Ingrese la dirección: ')

# Geolocalizamos la dirección para extraer las coordenadas GPS

averia = geolocator.geocode(direccion, timeout=None)
coche_coords = (averia.latitude, averia.longitude)

# Filtrado Espacial con (k-d Tree)

K = 5 # Elegimos el/los talleres mas cercanos en linea recta.

df1['Latitud'] = pd.to_numeric(df1['Latitud'], errors='coerce')
df1['Longitud'] = pd.to_numeric(df1['Longitud'], errors='coerce')
df1 = df1.dropna(subset=['Latitud', 'Longitud'])
coords_talleres = df1[['Latitud', 'Longitud']].values
tree = cKDTree(coords_talleres)

distancias_euclideas, indices_candidatos = tree.query(coche_coords, k=K)
candidatos_df = df1.iloc[np.atleast_1d(indices_candidatos)].copy()

# Inicializar el mapa vial

if K == 1:
    d = distancias_euclideas*100000
else:
    d = max(distancias_euclideas)*200000

G = ox.graph_from_point((coche_coords[0], coche_coords[1]), dist=d, network_type="drive")

# Mapear el coche al nodo más cercano del grafo
nodo_coche = ox.nearest_nodes(G, coche_coords[1], coche_coords[0])

# Mapear los K talleres candidatos a nodos del grafo
candidatos_df['nodo_grafo'] = ox.nearest_nodes(
    G,
    candidatos_df['Longitud'].values,
    candidatos_df['Latitud'].values
)

# Calcular la distancia real sobre la red viaria solo para los K candidatos
distancias_red = {}

for idx, row in candidatos_df.iterrows():
    nodo_taller = row['nodo_grafo']
    try:
        # Distancia en metros siguiendo las calles
        dist_metros = nx.shortest_path_length(
            G,
            source=nodo_coche,
            target=nodo_taller,
            weight='length'
        )
        distancias_red[row['Nombre del Taller']] = dist_metros
    except nx.NetworkXNoPath:
        # En caso de que el nodo esté aislado o no haya ruta posible
        distancias_red[row['Nombre del Taller']] = float('inf')

# Identificar el taller con la menor distancia en red
taller_ganador_id = min(distancias_red, key=distancias_red.get)
distancia_minima = distancias_red[taller_ganador_id]

# EXtraer coordenadas GPS del Taller más cercano
taller_ganador = candidatos_df[candidatos_df['Nombre del Taller'] == taller_ganador_id].iloc[0]
latitud_taller = taller_ganador['Latitud']
longitud_taller = taller_ganador['Longitud']

# Calcular la ruta
nodo_origen = ox.distance.nearest_nodes(G, X=averia.longitude, Y=averia.latitude)
nodo_destino = ox.distance.nearest_nodes(G, X=longitud_taller, Y=latitud_taller)

ruta = nx.shortest_path(G, source=nodo_origen, target=nodo_destino, weight="length")

# Extraer las coordenadas (lat, lon) de cada nodo de la ruta

coordenadas_ruta = []
for nodo in ruta:
    lat = G.nodes[nodo]["y"]
    lon = G.nodes[nodo]["x"]
    coordenadas_ruta.append([lat, lon])

# Crear el mapa base de Folium centrado en el inicio de la ruta

# Guardar tu clave API aquí
MI_API_KEY = "cb1_3kbu_1_961dfe3e10a17fd0bdcb4c97"

# 2. Elige el estilo de mapa que prefieras (descomenta solo uno):
# Opción A: CartoDB Positron (Fondo claro limpio)
# url_tiles =    f"https://cartocdn.com{{z}}/{{x}}/{{y}}.png?key={MI_API_KEY}"
# Opción B: CartoDB Dark Matter (Fondo oscuro elegante)
# url_tiles = f"https://basemaps.cartocdn.com/rastertiles/dark_all/{{z}}/{{x}}/{{y}}.png?key={MI_API_KEY}"

# Opción C: CartoDB Voyager (Fondo con más colores de calles)
url_tiles = f"https://basemaps.cartocdn.com/rastertiles/voyager/{{z}}/{{x}}/{{y}}.png?key={MI_API_KEY}"

# Crear el mapa pasando la URL personalizada y los créditos obligatorios
mapa = folium.Map(
    location=coordenadas_ruta[0],
    zoom_start=17,
    tiles=url_tiles,
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> &copy; <a href="https://carto.com/attributions">CARTO</a>',
)

# Dibujar la línea de la ruta en el mapa
folium.PolyLine(
    locations=coordenadas_ruta,
    color="blue",
    weight=5,
    opacity=0.8,
    tooltip="Ruta óptima",
).add_to(mapa)

# Añadir marcadores para el Origen y el Destino
folium.Marker(
    location=coordenadas_ruta[0],
    popup="Coche Averiado",
    icon=folium.Icon(color="green", icon="play"),
).add_to(mapa)

folium.Marker(
    location=coordenadas_ruta[-1],
    popup="Taller más cercano",
    icon=folium.Icon(color="red", icon="stop"),
).add_to(mapa)

# Mostrar mapa
mapa
